# 05 Associated factors

Future teammate work: analysis of associated factors; preserve observed values and document denominators.

## Function checks on known pairs (Member 3, 14 Nov 2026)

These cells check the functions in `src/associations.py` on pairs whose answer is known in advance. **They are not Q2 results.** The leading predictors for Q2 will be chosen from Member 3's model ranking, after which the association screen will be run.

All functions use observed pairs only (no imputation) and return `estimate`, `n_used` and `missing_pair_fraction`. Only the observed training split is read.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

sys.path.insert(0, str(Path("..").resolve() / "src"))
from associations import (
    correlation_ratio, cramers_v_bc, group_bootstrap, sparse_groups, spearman_assoc,
)

df = pd.read_csv("../data/processed/observed/training.csv")
print("Training split:", df.shape)

Training split: (308080, 41)


### Spearman: BMI vs WeightInKilograms

Expected: the value in Member 2's EDA (`results/tables/spearman_correlation.csv`, about 0.847) with n = 274,752 pairs (`results/tables/spearman_pair_counts.csv`).

In [2]:
eda_spearman = pd.read_csv("../results/tables/spearman_correlation.csv", index_col="variable")
eda_pairs = pd.read_csv("../results/tables/spearman_pair_counts.csv", index_col="variable")

bmi_weight = spearman_assoc(df["BMI"], df["WeightInKilograms"])
print(bmi_weight)
assert bmi_weight["n_used"] == eda_pairs.loc["BMI", "WeightInKilograms"] == 274752
assert abs(bmi_weight["estimate"] - eda_spearman.loc["BMI", "WeightInKilograms"]) < 1e-9
print("Matches the EDA: rho =", round(bmi_weight["estimate"], 3), "with n =", f"{bmi_weight['n_used']:,}")

{'estimate': 0.8466525103485081, 'n_used': 274752, 'missing_pair_fraction': 0.10817969358608148}
Matches the EDA: rho = 0.847 with n = 274,752


### Cramér's V (bias-corrected): a variable with itself, and with a shuffled copy

Expected: about 1 for `GeneralHealth` with itself, and about 0 for `GeneralHealth` with a randomly shuffled copy of itself (seed 42).

In [3]:
self_v = cramers_v_bc(df["GeneralHealth"], df["GeneralHealth"])
print("With itself:", {k: v for k, v in self_v.items() if k != "table"})
assert abs(self_v["estimate"] - 1) < 1e-9

rng = np.random.default_rng(42)
shuffled = df["GeneralHealth"].copy()
observed = shuffled.notna()
shuffled[observed] = rng.permutation(shuffled[observed].to_numpy())
shuffled_v = cramers_v_bc(df["GeneralHealth"], shuffled)
print("With shuffled copy:", {k: v for k, v in shuffled_v.items() if k != "table"})
assert shuffled_v["estimate"] < 0.01

With itself: {'estimate': 0.9999999999999999, 'n_used': 307335, 'missing_pair_fraction': 0.0024182030641392016}


With shuffled copy:

 {'estimate': 0.0007848742436874935, 'n_used': 307335, 'missing_pair_fraction': 0.0024182030641392016}


### Correlation ratio (eta): BMI by BMI group

`BMI_Group` uses Member 2's definition (`notebooks/03_eda.ipynb`, cell 68). The group is derived from BMI itself, so eta should be high.

In [4]:
bmi_group = pd.cut(
    df["BMI"],
    bins=[0, 18.5, 25, 30, 35, 40, float("inf")],
    labels=["Underweight", "Normal", "Overweight", "Obesity Class I", "Obesity Class II", "Obesity Class III"],
    right=False,
)
bmi_eta = correlation_ratio(bmi_group, df["BMI"])
print(bmi_eta)
assert bmi_eta["estimate"] > 0.8 and bmi_eta["n_used"] == 274752

{'estimate': 0.9490212584278763, 'n_used': 274752, 'missing_pair_fraction': 0.10817969358608148}


### A constant column gives NaN, not an error

In [5]:
constant = pd.Series(1.0, index=df.index)
constant_checks = {
    "spearman_assoc": spearman_assoc(constant, df["BMI"]),
    "cramers_v_bc": cramers_v_bc(constant, df["Sex"]),
    "correlation_ratio (constant response)": correlation_ratio(df["Sex"], constant),
}
for name, result in constant_checks.items():
    print(name, {k: v for k, v in result.items() if k != "table"})
    assert np.isnan(result["estimate"])

spearman_assoc {'estimate': nan, 'n_used': 274752, 'missing_pair_fraction': 0.10817969358608148}
cramers_v_bc {'estimate': nan, 'n_used': 308080, 'missing_pair_fraction': 0.0}
correlation_ratio (constant response) {'estimate': nan, 'n_used': 308080, 'missing_pair_fraction': 0.0}


### Sparse groups and the group bootstrap

`sparse_groups` should flag the Sex and HadDiabetes combination of Male with pregnancy-only diabetes (15 records; see the clarification in `03_eda.ipynb`). `group_bootstrap` resamples whole `duplicate_group` groups. Here it runs with 100 resamples to keep the notebook quick; the interval should contain the BMI–weight estimate above.

In [6]:
sex_diabetes = df["Sex"] + " | " + df["HadDiabetes"]
sex_diabetes_groups = sparse_groups(sex_diabetes, min_n=30)
display(sex_diabetes_groups)
flagged = sex_diabetes_groups.loc[sex_diabetes_groups["sparse"]]
assert list(flagged["category"]) == ["Male | Yes, but only during pregnancy (female)"]
assert flagged["n"].iloc[0] == 15

bmi_weight_boot = group_bootstrap(
    df[["BMI", "WeightInKilograms", "duplicate_group"]],
    lambda d: spearman_assoc(d["BMI"], d["WeightInKilograms"]),
    n_boot=100,
)
print(bmi_weight_boot)
assert bmi_weight_boot["ci_lower"] <= bmi_weight["estimate"] <= bmi_weight_boot["ci_upper"]

,category,n,sparse
0,"Male | Yes, but only during pregnancy (female)",15,True
1,"Female | Yes, but only during pregnancy (female)",2681,False
2,"Male | No, pre-diabetes or borderline diabetes",3118,False
3,"Female | No, pre-diabetes or borderline diabetes",3906,False
4,Male | Yes,20833,False
5,Female | Yes,21448,False
6,Male | No,120557,False
7,Female | No,134998,False


{'estimate': 0.8466525103485081, 'ci_lower': 0.8453842808265891, 'ci_upper': 0.8476217885275321, 'n_boot_valid': 100, 'n_groups': 307971}
